# 08 · Вмешательства в точке вызова: приблизиться к LoRA без обучения весов

**Цель** (JOURNAL, 2026-10-27/30). Диагностика 07: эффект LoRA передаётся через одну позицию — точку
вызова — на слоях 22+, а изменение, которое делает LoRA, — в основном постоянный сдвиг плюс линейная
функция состояния. Методы здесь — поправка в этой точке:

| метод | как | обучение |
|---|---|---|
| `linmap_lora` | аффинная поправка, подогнанная в закрытой форме к Δh LoRA | только у учителя (LoRA) |
| `linmap` | то же без учителя: цель — центроид верных состояний базы | **нет** |
| `reft` | LoReFT (2404.03592) в точке вызова | градиент только по вмешательству |
| `cast` | CAST (2409.05907): условие по PC1, вектор поведения | нет |

Данные — те же 110 промптов, что у вектора и LoRA; выбор на val; проверка на новом наборе переноса
(seed 202). Плюс проверка формы из 07 на слоях 22–26. Результат **разведочный**.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="act-methods-qwen3-1.7b", hypothesis="explore", notebook="notebooks/08_activation_methods.ipynb",
    notes="Вмешательства в точке вызова: linmap (с LoRA-учителем и без), LoReFT, CAST против LoRA.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "steering", "steering_gated", "lora",
                         "linmap_lora", "linmap", "reft", "cast"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 202,
        "transfer.episode_methods": ["baseline", "lora", "linmap_lora", "linmap", "reft"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

### Подбор на val

In [ ]:
for m in ("linmap_lora", "linmap", "reft", "cast"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, exp.state["fitted"][m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = [c for c in ("layer", "rank", "alpha", "cond_layer", "fire_rate", "train_r2", "train_seconds",
                        "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken") if c in t]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

### Проверка формы на слоях 22–26 (диагностика 07, без подстановки)

In [ ]:
diag = exp.lora_diagnostics(patch=False, form_layers=[22, 23, 24, 25, 26])
pd.DataFrame(diag["form_check"]).round(3)

### Новый набор переноса (seed 202): знакомые и новые типы требований

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

In [ ]:
rows = []
for g in ("seen", "unseen"):
    for name, c in exp.state["transfer"][g]["comparisons"].items():
        rows.append({"group": g, "comparison": name, "diff": c["mean_diff"],
                     "ci": f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", "excludes_zero": c["excludes_zero"]})
pd.DataFrame(rows)

### Цена

In [ ]:
f = exp.state["fitted"]
d = exp.backend.hidden_size
cost = [{"method": "steering (CAA)", "train": "среднее по 110 промптам", "seconds": None, "params": d}]
lr = f["lora"]["row"]
cost.append({"method": "lora", "train": "градиент по весам (копия fp32)", "seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m in ("linmap_lora", "linmap"):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        cost.append({"method": m, "train": "закрытая форма" + (" + учитель LoRA" if m == "linmap_lora" else ""),
                     "seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0), "params": 2 * d * r["rank"] + d})
r = f["reft"]["row"]
if r:
    cost.append({"method": "reft", "train": "градиент только по вмешательству", "seconds": r["train_seconds"],
                 "params": r["n_trainable"]})
if f["cast"]["row"]:
    cost.append({"method": "cast", "train": "PC1 + порог по F1", "seconds": None, "params": 2 * d + 1})
pd.DataFrame(cost)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))